<a href="https://colab.research.google.com/github/ojohnso3-oss/ojohnso3-INST414/blob/main/Lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline

In [ ]:
import pandas as pd
import json

from sklearn.cluster import KMeans

In [60]:
actor_name_map = {}
movie_actor_map = {}
actor_genre_map = {}


with open("/content/sample_data/imdb_movies_2000to2022.prolific.json", "r") as in_file:
    for line in in_file:

        # Read the movie on this line and parse its json
        this_movie = json.loads(line)

        # Add all actors to the id->name map
        for actor_id,actor_name in this_movie['actors']:
            actor_name_map[actor_id] = actor_name

        # For each actor, add this movie's genres to that actor's list
        for actor_id,actor_name in this_movie['actors']:
            this_actors_genres = actor_genre_map.get(actor_id, {})

            # Increment the count of genres for this actor
            for g in this_movie["genres"]:
                this_actors_genres[g] = this_actors_genres.get(g, 0) + 1

            # Update the map
            actor_genre_map[actor_id] = this_actors_genres

        # Finished with this film
        movie_actor_map[this_movie["imdb_id"]] = ({
            "movie": this_movie["title"],
            "actors": set([item[0] for item in this_movie['actors']]),
            "genres": this_movie["genres"]
        })

In [61]:
actor_genre_map['nm0413168']

{'Comedy': 7,
 'Fantasy': 3,
 'Romance': 5,
 'Action': 14,
 'Adventure': 11,
 'Sci-Fi': 10,
 'Crime': 6,
 'Thriller': 2,
 'Animation': 4,
 'Drama': 12,
 'Mystery': 5,
 'Biography': 4,
 'Musical': 2,
 'History': 1}

In [ ]:
# Get all actors as an index for a dataframe
index = actor_genre_map.keys()

# Get the genre-counts for each actor in the index
rows = [actor_genre_map[k] for k in index]

# Create the data frame from these rows, with the actors as index
df = pd.DataFrame(rows, index=index)

# Fill NAs with zero, as NA means the actor has not starred in that genre
df = df.fillna(0)

df

In [ ]:
df = pd.read_csv("/content/sample_data/imdb_movies_2000to2022.actorXgenre.csv", index_col="actor_id")


In [ ]:
df

,Comedy,Fantasy,Romance,Drama,Mystery,Thriller,Action,Biography,Crime,War,...,Horror,Documentary,Sport,News,Family,Music,Unnamed: 22,Western,Short,Reality-TV
actor_id,,,,,,,,,,,,,,,,,,,,,
nm0000212,7.0,1.0,6.0,6.0,1.0,2.0,1.0,1.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0413168,7.0,3.0,5.0,12.0,5.0,2.0,14.0,4.0,6.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0000630,8.0,2.0,6.0,14.0,2.0,3.0,4.0,5.0,1.0,1.0,...,3.0,7.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0005227,10.0,1.0,2.0,2.0,0.0,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
nm0864851,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
nm9504284,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm10592896,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm7216750,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from sklearn.cluster import KMeans

In [ ]:
df = df.loc[:, df.columns.str.strip() != ""]

In [ ]:
model = KMeans(n_clusters=8, random_state=42, n_init=10)
cluster_labels = model.fit_predict(df)

In [65]:
cluster_model.fit(df)

KMeans()

In [56]:
actor_cluster_df = pd.DataFrame(cluster_labels, index=df.index, columns=["cluster"])

In [58]:
counts = actor_cluster_df["cluster"].value_counts().sort_index()
for cid, cnt in counts.items():
    print(f"  Cluster {cid}: {cnt} actors")

  Cluster 0: 2556 actors
  Cluster 1: 29329 actors
  Cluster 2: 193 actors
  Cluster 3: 16 actors
  Cluster 4: 261 actors
  Cluster 5: 249 actors
  Cluster 6: 135 actors
  Cluster 7: 870 actors


In [64]:
for cluster, actors in actor_cluster_df.groupby("cluster"):
    print("Cluster:", cluster, "Size:", actors.shape[0])

    for a_id in actors.sample(5).index:
        print("\t", a_id, actor_name_map[a_id])

    print()

Cluster: 0 Size: 2556
	 nm0802183 Neetu Singh
	 nm0000986 Bryan Brown
	 nm0186120 Paul Cram
	 nm0261805 Erik Estrada
	 nm3241414 Penelope Mitchell

Cluster: 1 Size: 29329
	 nm3988475 Paige A. Flash
	 nm0947490 J.R. Yenque
	 nm0226950 Pamela Dillman
	 nm1936266 Harli Ames
	 nm1227811 Jennifer Howie

Cluster: 2 Size: 193
	 nm0001218 Sean Patrick Flanery
	 nm3008388 Michael Fredianelli
	 nm0006763 Jackie Shroff
	 nm1352774 Jose Rosete
	 nm1527905 Toby Kebbell

Cluster: 3 Size: 16
	 nm0474774 Akshay Kumar
	 nm0000246 Bruce Willis
	 nm0695177 Prakash Raj
	 nm0000616 Eric Roberts
	 nm0005458 Jason Statham

Cluster: 4 Size: 261
	 nm0005188 James Marsden
	 nm0000182 Jennifer Lopez
	 nm1229940 Katrina Kaif
	 nm0000511 Shirley MacLaine
	 nm0000337 Rachael Leigh Cook

Cluster: 5 Size: 249
	 nm0004569 Sanjay Dutt
	 nm0095478 Mark Boone Junior
	 nm1209966 Oscar Isaac
	 nm0268199 Colin Farrell
	 nm0000204 Natalie Portman

Cluster: 6 Size: 135
	 nm1767095 Stacey T. Gillespie
	 nm0879891 Helene Udy
	 